In [1]:
import os, sys
import numpy as np
from functools import partial
from multiprocessing.pool import Pool

from sedflow import obs as Obs
from sedflow import train as Train

from provabgs import infer as Infer
from provabgs import models as Models

/home/chhahn/projects/provabgs/src/provabgs/models.py:23: UserWarning: import error with fsps; only use emulators
  warnings.warn('import error with fsps; only use emulators')


In [2]:
sample = 'toy'
itrain = 2
nhidden = 500
nblocks = 15

## compile NSA failures

In [3]:
# u, g, r, i, z, sigma_u, sigma_g, sigma_r, sigma_i, sigma_z, redshift
y_nsa = Obs.load_nsa_data(test_set=False)

ffail = os.path.join('/scratch/network/chhahn/sedflow/nsa_fail/', 'fail.igals.npy')
if not os.path.isfile(ffail): 
    igals = []
    for ichunk in range(34):
        fpost = os.path.join(Train.data_dir(), 'anpe_thetaunt_magsigz.%s.%ix%i.%i.nsa%iof34.samples.npy' % (sample, nhidden, nblocks, itrain, ichunk))
        if not os.path.isfile(fpost): continue
        post = np.load(fpost)
        fail = (np.sum(np.sum(post, axis=2), axis=1) == 0)
        igals.append(np.arange(y_nsa.shape[0])[ichunk*1000:(ichunk+1)*1000][fail])

    igals = np.concatenate(igals)
    
    np.save(ffail, igals)
else: 
    igals = np.load(ffail)

In [4]:
# convert to flux
y_flux = Train.mag2flux(y_nsa[:,:5])
y_ivar = Train.sigma_mag2flux(y_nsa[:,5:10], y_nsa[:,:5]) ** -2
y_zred = y_nsa[:,-1]

In [5]:
# SPS parameter priors
prior_sps = Infer.load_priors([
        Infer.UniformPrior(7., 12.5, label='sed'),
        Infer.FlatDirichletPrior(4, label='sed'),           # flat dirichilet priors
        Infer.UniformPrior(0., 1., label='sed'),            # burst fraction
        Infer.UniformPrior(1e-2, 13.27, label='sed'),    # tburst
        Infer.LogUniformPrior(4.5e-5, 1.5e-2, label='sed'), # log uniform priors on ZH coeff
        Infer.LogUniformPrior(4.5e-5, 1.5e-2, label='sed'), # log uniform priors on ZH coeff
        Infer.UniformPrior(0., 3., label='sed'),        # uniform priors on dust1
        Infer.UniformPrior(0., 3., label='sed'),        # uniform priors on dust2
        Infer.UniformPrior(-2., 1., label='sed')     # uniform priors on dust_index
    ])

# SPS model
m_sps = Models.NMF(burst=True, emulator=True)

input parameters : logmstar, beta1_sfh, beta2_sfh, beta3_sfh, beta4_sfh, fburst, tburst, gamma1_zh, gamma2_zh, dust1, dust2, dust_index


In [6]:
def run_mcmc(i_obs, niter=3000):
    # desi MCMC object
    nsa_mcmc = Infer.nsaMCMC(model=m_sps, prior=prior_sps)

    fmcmc = os.path.join('/scratch/network/chhahn/sedflow/nsa_fail',
            'mcmc.nsa.%i.hdf5' % i_obs)
    print(fmcmc)

    # run MCMC
    zeus_chain = nsa_mcmc.run(
            bands='sdss', # u, g, r, i, z
            photo_obs=y_flux[i_obs],
            photo_ivar_obs=y_ivar[i_obs],
            zred=y_zred[i_obs],
            vdisp=0.,
            sampler='zeus',
            nwalkers=30,
            burnin=0,
            opt_maxiter=2000,
            niter=niter,
            progress=True,
            writeout=fmcmc)
    return None

In [7]:
run_mcmc(igals[0], niter=3000)

/scratch/network/chhahn/sedflow/nsa_fail/mcmc.nsa.1078.hdf5


Initialising ensemble of 30 walkers...
Sampling progress :   4%|▍         | 134/3000 [01:48<32:41,  1.46it/s]

zeus: Exception while calling your likelihood function:
  params: [9.60939334e+00 5.13969747e-01 5.63776346e-01 1.36352922e-01
 1.00258311e-01 8.17552983e+00 7.06229778e-04 6.40831246e-03
 4.06371238e-01 3.06251563e-01 2.09701698e-01]
  args: (None, None, None, array([ 35.560593,  91.709236, 139.84373 , 169.70816 , 194.97598 ],
      dtype=float32), array([0.22200741, 0.30760366, 0.00446658, 0.05008115, 0.01728794],
      dtype=float32), 0.0351418, 0.0)
  kwargs: {'resolution': None, 'mask': None, 'filters': <speclite.filters.FilterSequence object at 0x7fcfa69c2d10>, 'obs_data_type': 'photo'}
  exception:


Traceback (most recent call last):
  File "/home/chhahn/.conda/envs/torch-env/lib/python3.7/site-packages/zeus/fwrapper.py", line 24, in __call__
    return self.f(x, *self.args, **self.kwargs)
  File "/home/chhahn/projects/provabgs/src/provabgs/infer.py", line 70, in lnPost
    lnlike = self.lnLike(ttheta, *args, debug=debug, **kwargs)
  File "/home/chhahn/projects/provabgs/src/provabgs/infer.py", line 631, in lnLike
    wavelength=wave_obs, resolution=resolution, filters=filters)
  File "/home/chhahn/projects/provabgs/src/provabgs/models.py", line 103, in sed
    wave_rest, lum_ssp = self._sps_model(_tt, _tage)
  File "/home/chhahn/projects/provabgs/src/provabgs/models.py", line 308, in _emu
    lum_burst = np.exp(self._emu_burst(tt))
  File "/home/chhahn/projects/provabgs/src/provabgs/models.py", line 515, in _emu_burst
    return self._emu_burst_nn(tt)
  File "/home/chhahn/projects/provabgs/src/provabgs/models.py", line 532, in _emu_burst_nn
    pca_shift_, pca_scale_, pca_transfor

KeyboardInterrupt: 